# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. 

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display metadata contents
print(f"Dataset name: {getattr(dataset.metadata, 'name', '')}")
print(f"Description: {getattr(dataset.metadata, 'description', '')}")
print(f"License: {getattr(dataset.metadata, 'license', '')}")
print(f"Version: {getattr(dataset.metadata, 'version', '')}")

## 2. Data Overview
Explore available record sets and fields. All entities are referenced by their Croissant schema `@id`.

List available record sets and detail their fields and columns by ID.

In [ ]:
# List all record set IDs from the dataset
record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []
print(f"Found {len(record_sets)} record set(s):")
record_set_ids = []
for i, rs in enumerate(record_sets):
    rs_id = getattr(rs, '@id', '')
    record_set_ids.append(rs_id)
    print(f"  {i+1}. @id: {rs_id}")
    # List field @id for each record set
    field_objs = getattr(rs, 'field', [])
    if field_objs and not isinstance(field_objs, list):
        field_objs = [field_objs]
    print("    Fields:")
    for field in field_objs:
        print(f"      - @id: {getattr(field, '@id', field)}")

## 3. Data Extraction
Load data from a specific record set using its `@id`. Reference fields and columns by their `@id` for clarity and schema consistency.

Below, we extract all record sets and load each into a DataFrame for further exploration.

In [ ]:
# Prepare and load record sets into pandas DataFrames
dataframes = {}
if record_set_ids:
    for rec_id in record_set_ids:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Record set @id: {rec_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
else:
    print("No record sets were found in the dataset's metadata.")

## 4. Exploratory Data Analysis (EDA)
Process and summarize the data using typical EDA operations. Use field `@id`s when referencing columns, and demonstrate actions such as filtering, normalization, and grouping.

In [ ]:
# For demonstration, select the first available record set and a likely numeric field
selected_rs_id = None
if dataframes:
    selected_rs_id = list(dataframes.keys())[0]
    df = dataframes[selected_rs_id]
    # Try to auto-detect a numeric field (float or int columns)
    numeric_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()
    print(f"Detected numeric fields: {numeric_fields}")
    if numeric_fields:
        numeric_field_id = numeric_fields[0]

        # Filter records with value above an arbitrary threshold
        threshold = df[numeric_field_id].mean() if len(df) > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the selected numeric field for filtered records
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # If any potential grouping/categorical field exists, group by it
        possible_group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if possible_group_fields:
            group_field_id = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped (by {group_field_id}) mean values:")
            display(grouped_df.head())
    else:
        print("No numeric fields detected in the chosen record set.")
else:
    print("No DataFrames available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the distribution of the detected numeric field (by field `@id`) in the selected record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'df' in locals() and numeric_fields:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field_id}")
    ax.set_xlabel(numeric_field_id)
    ax.set_ylabel("Count")
    plt.show()
else:
    print("No suitable numeric field to plot.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and examine the FAIR² mass spectrometry dataset by referencing all entities with their Croissant `@id`. We previewed dataset structure, performed basic EDA including filtering and normalization, and visualized field distributions for further scientific insights.